In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().absolute().parent))

In [2]:
import pandas as pd

from src.data_models.caravanify import Caravanify, CaravanifyConfig

---

In [3]:
PATH_TO_CLUSTER_ASSIGNED = Path(
    "/Users/cooper/Desktop/CAMELS-CH/clustering_results/cluster_assignments_shifted.csv"
)

df = pd.read_csv(PATH_TO_CLUSTER_ASSIGNED)


def get_ids_for_cluster(df, cluster_id):
    """
    Get the ids for a given cluster id
    """
    return df[df["cluster"] == cluster_id]["gauge_id"].values


def fileter_ids_by_country(ids, country_prefix):
    """
    Returns the ids the start with the given country prefix
    """
    return [id for id in ids if id.startswith(country_prefix)]

In [4]:
df

,gauge_id,cluster
0,CH_2009,14
1,CH_2011,14
2,CH_2016,3
3,CH_2018,14
4,CH_2019,14
...,...,...
1025,USA_14309500,1
1026,USA_14316700,1
1027,USA_14325000,1
1028,USA_14362250,1


---

In [5]:
CA_config = CaravanifyConfig(
    attributes_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CA/post_processed/attributes",
    timeseries_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CA/post_processed/timeseries/csv",
    shapefile_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CA/post_processed/shapefiles",
    human_influence_path="/Users/cooper/Desktop/CAMELS-CH/src/human_influence_index/results/human_influence_classification.csv",
    gauge_id_prefix="CA",
    use_hydroatlas_attributes=True,
    use_caravan_attributes=True,
    use_other_attributes=True,
)

CH_config = CaravanifyConfig(
    attributes_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CH/post_processed/attributes",
    timeseries_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CH/post_processed/timeseries/csv",
    shapefile_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CH/post_processed/shapefiles",
    human_influence_path="/Users/cooper/Desktop/CAMELS-CH/src/human_influence_index/results/human_influence_classification.csv",
    gauge_id_prefix="CH",
    use_hydroatlas_attributes=True,
    use_caravan_attributes=True,
    use_other_attributes=True,
)

CL_config = CaravanifyConfig(
    attributes_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CL/post_processed/attributes",
    timeseries_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CL/post_processed/timeseries/csv",
    shapefile_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/CL/post_processed/shapefiles",
    human_influence_path="/Users/cooper/Desktop/CAMELS-CH/src/human_influence_index/results/human_influence_classification.csv",
    gauge_id_prefix="CL",
    use_hydroatlas_attributes=True,
    use_caravan_attributes=True,
    use_other_attributes=True,
)

USA_config = CaravanifyConfig(
    attributes_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/USA/post_processed/attributes",
    timeseries_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/USA/post_processed/timeseries/csv",
    shapefile_dir="/Users/cooper/Desktop/CAMELS-CH/data/CARAVANIFY/USA/post_processed/shapefiles",
    human_influence_path="/Users/cooper/Desktop/CAMELS-CH/src/human_influence_index/results/human_influence_classification.csv",
    gauge_id_prefix="USA",
    use_hydroatlas_attributes=True,
    use_caravan_attributes=True,
    use_other_attributes=True,
)

CH_caravan = Caravanify(CH_config)
CL_caravan = Caravanify(CL_config)
USA_caravan = Caravanify(USA_config)

In [6]:
CH_static = CH_caravan.get_static_attributes()
CL_static = CL_caravan.get_static_attributes()
USA_static = USA_caravan.get_static_attributes()

In [10]:
clusters = [i for i in range(15)]
all_clusters_data = []  # Will hold dataframes from all clusters

for cluster in clusters:
    df_copy = df.copy()
    cluster_ids = get_ids_for_cluster(df_copy, cluster)

    CH_ids = fileter_ids_by_country(cluster_ids, "CH")
    CL_ids = fileter_ids_by_country(cluster_ids, "CL")
    USA_ids = fileter_ids_by_country(cluster_ids, "USA")

    CH_caravan.load_stations(CH_ids)
    CL_caravan.load_stations(CL_ids)
    USA_caravan.load_stations(USA_ids)

    statics_of_interest = [
        "p_mean",
        "pet_mean_FAO_PM",
        "frac_snow",
        "aridity_FAO_PM",
        "seasonality_FAO_PM",
        "area",
        "ele_mt_sav",
    ]
    CH_static = CH_caravan.get_static_attributes()[statics_of_interest]
    CL_static = CL_caravan.get_static_attributes()[statics_of_interest]
    USA_static = USA_caravan.get_static_attributes()[statics_of_interest]

    all_static = pd.concat([CH_static, CL_static, USA_static], axis=0)
    all_static["cluster"] = cluster

    # Count stations for each country
    num_stations = len(cluster_ids)

    all_static["p_mean"] = all_static["p_mean"] * 365
    all_static["pet_mean_FAO_PM"] = all_static["pet_mean_FAO_PM"] * 365

    all_clusters_data.append(all_static)

combined_df_copy = pd.concat(all_clusters_data, axis=0)

results = pd.DataFrame(index=clusters)

# Calculate statistics for attributes
for attr in statics_of_interest:
    stats = combined_df_copy.groupby("cluster")[attr].agg(["mean", "std"])

    formatted_stats = stats.apply(
        lambda row: f"{row['mean']:.1f} ± {row['std']:.1f}", axis=1
    )

    results[attr] = formatted_stats

# Add number of stations column separately
for cluster in clusters:
    df_copy = df.copy()
    cluster_ids = get_ids_for_cluster(df_copy, cluster)
    results.at[cluster, "number_of_stations"] = len(cluster_ids)

print(results)

# Optional: Save results to CSV
# results.to_csv('cluster_statistics.csv')

             p_mean pet_mean_FAO_PM  frac_snow aridity_FAO_PM  \
0     928.5 ± 276.6   723.2 ± 139.1  0.3 ± 0.1      0.9 ± 0.5   
1    1537.8 ± 571.9   851.6 ± 155.2  0.0 ± 0.0      0.7 ± 0.7   
2    1308.6 ± 632.4   704.5 ± 272.9  0.2 ± 0.2      0.9 ± 0.9   
3    1263.3 ± 478.4   705.5 ± 181.4  0.2 ± 0.2      0.7 ± 0.4   
4    1206.4 ± 294.1   767.2 ± 164.1  0.2 ± 0.1      0.7 ± 0.4   
5     1127.1 ± 87.2   1204.5 ± 64.0  0.0 ± 0.0      1.1 ± 0.1   
6    1202.1 ± 157.1   907.0 ± 133.0  0.0 ± 0.1      0.8 ± 0.3   
7     671.3 ± 167.9  1306.3 ± 146.7  0.0 ± 0.0      2.1 ± 0.7   
8    1709.8 ± 464.0   700.1 ± 136.5  0.1 ± 0.1      0.5 ± 0.3   
9    1009.0 ± 369.2  1193.6 ± 241.1  0.1 ± 0.2      1.4 ± 0.5   
10  1608.1 ± 1228.4   610.7 ± 332.8  0.2 ± 0.2      1.1 ± 1.4   
11    867.8 ± 555.3  1070.5 ± 125.7  0.0 ± 0.0      1.7 ± 1.1   
12    903.0 ± 172.5  1000.1 ± 137.5  0.1 ± 0.1      1.2 ± 0.3   
13    808.9 ± 317.0   730.4 ± 154.3  0.5 ± 0.2      1.1 ± 0.5   
14   1392.4 ± 500.8   489

In [8]:
# Columns in CH_static: ['gauge_id', 'area', 'country', 'gauge_lat', 'gauge_lon', 'gauge_name', 'aet_mm_s01', 'aet_mm_s02', 'aet_mm_s03', 'aet_mm_s04', 'aet_mm_s05', 'aet_mm_s06', 'aet_mm_s07', 'aet_mm_s08', 'aet_mm_s09', 'aet_mm_s10', 'aet_mm_s11', 'aet_mm_s12', 'aet_mm_syr', 'area_fraction_used_for_aggregation', 'ari_ix_sav', 'cls_cl_smj', 'cly_pc_sav', 'clz_cl_smj', 'cmi_ix_s01', 'cmi_ix_s02', 'cmi_ix_s03', 'cmi_ix_s04', 'cmi_ix_s05', 'cmi_ix_s06', 'cmi_ix_s07', 'cmi_ix_s08', 'cmi_ix_s09', 'cmi_ix_s10', 'cmi_ix_s11', 'cmi_ix_s12', 'cmi_ix_syr', 'crp_pc_sse', 'dis_m3_pmn', 'dis_m3_pmx', 'dis_m3_pyr', 'dor_pc_pva', 'ele_mt_sav', 'ele_mt_smn', 'ele_mt_smx', 'ero_kh_sav', 'fec_cl_smj', 'fmh_cl_smj', 'for_pc_sse', 'gdp_ud_sav', 'gdp_ud_ssu', 'gla_pc_sse', 'glc_cl_smj', 'glc_pc_s01', 'glc_pc_s02', 'glc_pc_s03', 'glc_pc_s04', 'glc_pc_s05', 'glc_pc_s06', 'glc_pc_s07', 'glc_pc_s08', 'glc_pc_s09', 'glc_pc_s10', 'glc_pc_s11', 'glc_pc_s12', 'glc_pc_s13', 'glc_pc_s14', 'glc_pc_s15', 'glc_pc_s16', 'glc_pc_s17', 'glc_pc_s18', 'glc_pc_s19', 'glc_pc_s20', 'glc_pc_s21', 'glc_pc_s22', 'gwt_cm_sav', 'hdi_ix_sav', 'hft_ix_s09', 'hft_ix_s93', 'inu_pc_slt', 'inu_pc_smn', 'inu_pc_smx', 'ire_pc_sse', 'kar_pc_sse', 'lit_cl_smj', 'lka_pc_sse', 'lkv_mc_usu', 'nli_ix_sav', 'pac_pc_sse', 'pet_mm_s01', 'pet_mm_s02', 'pet_mm_s03', 'pet_mm_s04', 'pet_mm_s05', 'pet_mm_s06', 'pet_mm_s07', 'pet_mm_s08', 'pet_mm_s09', 'pet_mm_s10', 'pet_mm_s11', 'pet_mm_s12', 'pet_mm_syr', 'pnv_cl_smj', 'pnv_pc_s01', 'pnv_pc_s02', 'pnv_pc_s03', 'pnv_pc_s04', 'pnv_pc_s05', 'pnv_pc_s06', 'pnv_pc_s07', 'pnv_pc_s08', 'pnv_pc_s09', 'pnv_pc_s10', 'pnv_pc_s11', 'pnv_pc_s12', 'pnv_pc_s13', 'pnv_pc_s14', 'pnv_pc_s15', 'pop_ct_usu', 'ppd_pk_sav', 'pre_mm_s01', 'pre_mm_s02', 'pre_mm_s03', 'pre_mm_s04', 'pre_mm_s05', 'pre_mm_s06', 'pre_mm_s07', 'pre_mm_s08', 'pre_mm_s09', 'pre_mm_s10', 'pre_mm_s11', 'pre_mm_s12', 'pre_mm_syr', 'prm_pc_sse', 'pst_pc_sse', 'rdd_mk_sav', 'rev_mc_usu', 'ria_ha_usu', 'riv_tc_usu', 'run_mm_syr', 'sgr_dk_sav', 'slp_dg_sav', 'slt_pc_sav', 'snd_pc_sav', 'snw_pc_s01', 'snw_pc_s02', 'snw_pc_s03', 'snw_pc_s04', 'snw_pc_s05', 'snw_pc_s06', 'snw_pc_s07', 'snw_pc_s08', 'snw_pc_s09', 'snw_pc_s10', 'snw_pc_s11', 'snw_pc_s12', 'snw_pc_smx', 'snw_pc_syr', 'soc_th_sav', 'swc_pc_s01', 'swc_pc_s02', 'swc_pc_s03', 'swc_pc_s04', 'swc_pc_s05', 'swc_pc_s06', 'swc_pc_s07', 'swc_pc_s08', 'swc_pc_s09', 'swc_pc_s10', 'swc_pc_s11', 'swc_pc_s12', 'swc_pc_syr', 'tbi_cl_smj', 'tec_cl_smj', 'tmp_dc_s01', 'tmp_dc_s02', 'tmp_dc_s03', 'tmp_dc_s04', 'tmp_dc_s05', 'tmp_dc_s06', 'tmp_dc_s07', 'tmp_dc_s08', 'tmp_dc_s09', 'tmp_dc_s10', 'tmp_dc_s11', 'tmp_dc_s12', 'tmp_dc_smn', 'tmp_dc_smx', 'tmp_dc_syr', 'urb_pc_sse', 'wet_cl_smj', 'wet_pc_s01', 'wet_pc_s02', 'wet_pc_s03', 'wet_pc_s04', 'wet_pc_s05', 'wet_pc_s06', 'wet_pc_s07', 'wet_pc_s08', 'wet_pc_s09', 'wet_pc_sg1', 'wet_pc_sg2', 'aridity_ERA5_LAND', 'aridity_FAO_PM', 'frac_snow', 'high_prec_dur', 'high_prec_freq', 'low_prec_dur', 'low_prec_freq', 'moisture_index_ERA5_LAND', 'moisture_index_FAO_PM', 'p_mean', 'pet_mean_ERA5_LAND', 'pet_mean_FAO_PM', 'seasonality_ERA5_LAND', 'seasonality_FAO_PM']
